In [1]:
import pandas as pd
df = pd.read_csv("../processed/films_clean.csv")


In [2]:
## must make some assumptions to estimate share of box office/downstream revenue

ASSUMPTIONS = {
    "k_wide":     1.0,   # marketing (P&A) as multiple of budget, wide release
    "k_platform": 0.5,   # marketing multiple, limited/platform release
    "r_dom":      0.50,  # studio's share of domestic box office (rental rate)
    "r_intl":     0.40,  # studio's share of international box office
    "a":          0.5,   # downstream/ancillary revenue as fraction of theatrical
}

In [3]:
WIDE_RELEASE_BUDGET = 15_000_000   ## this is relatively arbitrary, but we need some threshold
df["is_wide_release"] = df["budget_usd"] >= WIDE_RELEASE_BUDGET
df["is_wide_release"].value_counts()

is_wide_release
False    181
True      34
Name: count, dtype: int64

In [4]:
## costing: Cost = production budget + marketing

def compute_cost(row, a):
    budget = row["budget_usd"]
    k = a["k_wide"] if row["is_wide_release"] else a["k_platform"]
    pna = k * budget                 # modelled marketing spend
    return budget + pna

df["cost_usd"] = df.apply(lambda r: compute_cost(r, ASSUMPTIONS), axis=1)
df

,film_id,imdb_id,title,release_date,release_year,runtime_min,genres,budget_usd,domestic_gross_usd,intl_gross_usd,...,resolution_confidence,missing_fields,flag_no_budget,flag_no_revenue,flag_still_in_release,flag_distribution_only,in_core,primary_genre,is_wide_release,cost_usd
0,124461.0,tt2044729,A Glimpse Inside the Mind of Charles Swan III,2013-02-08,2013,86.0,Comedy|Drama,12000000.0,45350.0,165215.0,...,search_exact,NaN,False,False,False,False,True,Comedy,False,18000000.0
1,122081.0,tt2101441,Spring Breakers,2013-03-06,2013,94.0,Drama|Crime,5000000.0,14124284.0,17891843.0,...,search_exact,NaN,False,False,False,False,True,Drama,False,7500000.0
2,96936.0,tt2132285,The Bling Ring,2013-06-12,2013,90.0,Drama|Crime,15000000.0,5845732.0,13300000.0,...,search_exact,NaN,False,False,False,False,True,Drama,True,30000000.0
3,157386.0,tt1714206,The Spectacular Now,2013-08-02,2013,95.0,Comedy|Drama|Romance,2500000.0,6854611.0,0.0,...,search_exact,NaN,False,False,False,False,True,Comedy,False,3750000.0
4,181886.0,tt2316411,Enemy,2014-03-14,2014,91.0,Thriller|Mystery,3500000.0,1008726.0,2388000.0,...,search_exact,NaN,False,False,False,False,True,Thriller,False,5250000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210,1325734.0,tt33071426,The Drama,2026-04-01,2026,105.0,Romance|Comedy|Drama,28000000.0,48062006.0,84165787.0,...,discover_title_match,rt_critics,False,False,True,False,True,Other,True,56000000.0
211,1102883.0,tt27200708,Mother Mary,2026-04-16,2026,112.0,Music|Drama|Thriller,20000000.0,NaN,NaN,...,discover_title_match,"domestic_gross_usd,rt_critics,awards_raw",False,False,True,False,True,Other,True,40000000.0
212,1083381.0,tt26657236,Backrooms,2026-05-27,2026,111.0,Horror|Mystery|Science Fiction,10000000.0,190480127.0,184416225.0,...,discover_title_match,rt_critics,False,False,True,False,True,Horror,False,15000000.0
213,1284465.0,tt32273171,The Death of Robin Hood,2026-06-18,2026,123.0,Adventure|Drama|Thriller|Action,25000000.0,NaN,NaN,...,search_exact,"domestic_gross_usd,metascore,rt_critics,awards...",False,False,True,False,True,Other,True,50000000.0


In [5]:
## now we need to find out the revenue, this considers more

def compute_revenue(row, a):
    dom, intl, ww = row["domestic_gross_usd"], row["intl_gross_usd"], row["worldwide_gross_usd"]
    if pd.isna(dom) or pd.isna(intl):
        ## 6 core films have a worldwide gross but no dom/intl split — use a
        ## blended rental rate on worldwide rather than NaN-ing them out of the sample
        theatrical = ww * (a["r_dom"] + a["r_intl"]) / 2
    else:
        ## studio's share of the box office
        ## (negative intl is a data artefact where OMDb domestic > TMDB worldwide; a gross can't be < 0)
        theatrical = dom * a["r_dom"] + max(intl, 0) * a["r_intl"]
    ## downstream: home video, TV, streaming licensing
    total = theatrical * (1 + a["a"])
    return total

df["studio_revenue_usd"] = df.apply(lambda r: compute_revenue(r, ASSUMPTIONS), axis=1)
df.head()

,film_id,imdb_id,title,release_date,release_year,runtime_min,genres,budget_usd,domestic_gross_usd,intl_gross_usd,...,missing_fields,flag_no_budget,flag_no_revenue,flag_still_in_release,flag_distribution_only,in_core,primary_genre,is_wide_release,cost_usd,studio_revenue_usd
0,124461.0,tt2044729,A Glimpse Inside the Mind of Charles Swan III,2013-02-08,2013,86.0,Comedy|Drama,12000000.0,45350.0,165215.0,...,NaN,False,False,False,False,True,Comedy,False,18000000.0,133141.50
1,122081.0,tt2101441,Spring Breakers,2013-03-06,2013,94.0,Drama|Crime,5000000.0,14124284.0,17891843.0,...,NaN,False,False,False,False,True,Drama,False,7500000.0,21328318.80
2,96936.0,tt2132285,The Bling Ring,2013-06-12,2013,90.0,Drama|Crime,15000000.0,5845732.0,13300000.0,...,NaN,False,False,False,False,True,Drama,True,30000000.0,12364299.00
3,157386.0,tt1714206,The Spectacular Now,2013-08-02,2013,95.0,Comedy|Drama|Romance,2500000.0,6854611.0,0.0,...,NaN,False,False,False,False,True,Comedy,False,3750000.0,5140958.25
4,181886.0,tt2316411,Enemy,2014-03-14,2014,91.0,Thriller|Mystery,3500000.0,1008726.0,2388000.0,...,NaN,False,False,False,False,True,Thriller,False,5250000.0,2189344.50


In [6]:
import numpy as np

df["profit_usd"]       = df["studio_revenue_usd"] - df["cost_usd"]
df["roi"]              = df["profit_usd"] / df["cost_usd"]        # e.g. 1.5 = +150%
df["revenue_multiple"] = df["studio_revenue_usd"] / df["cost_usd"]
df["log_multiple"]     = np.log(df["revenue_multiple"])

df.head()

,film_id,imdb_id,title,release_date,release_year,runtime_min,genres,budget_usd,domestic_gross_usd,intl_gross_usd,...,flag_distribution_only,in_core,primary_genre,is_wide_release,cost_usd,studio_revenue_usd,profit_usd,roi,revenue_multiple,log_multiple
0,124461.0,tt2044729,A Glimpse Inside the Mind of Charles Swan III,2013-02-08,2013,86.0,Comedy|Drama,12000000.0,45350.0,165215.0,...,False,True,Comedy,False,18000000.0,133141.50,-17866858.50,-0.992603,0.007397,-4.906715
1,122081.0,tt2101441,Spring Breakers,2013-03-06,2013,94.0,Drama|Crime,5000000.0,14124284.0,17891843.0,...,False,True,Drama,False,7500000.0,21328318.80,13828318.80,1.843776,2.843776,1.045133
2,96936.0,tt2132285,The Bling Ring,2013-06-12,2013,90.0,Drama|Crime,15000000.0,5845732.0,13300000.0,...,False,True,Drama,True,30000000.0,12364299.00,-17635701.00,-0.587857,0.412143,-0.886384
3,157386.0,tt1714206,The Spectacular Now,2013-08-02,2013,95.0,Comedy|Drama|Romance,2500000.0,6854611.0,0.0,...,False,True,Comedy,False,3750000.0,5140958.25,1390958.25,0.370922,1.370922,0.315484
4,181886.0,tt2316411,Enemy,2014-03-14,2014,91.0,Thriller|Mystery,3500000.0,1008726.0,2388000.0,...,False,True,Thriller,False,5250000.0,2189344.50,-3060655.50,-0.582982,0.417018,-0.874626


In [7]:
core = df[df["in_core"]].copy()
core[["title", "budget_usd", "studio_revenue_usd", "roi", "revenue_multiple"]] \
    .sort_values("roi", ascending=False).head(10)

,title,budget_usd,studio_revenue_usd,roi,revenue_multiple
90,The Farewell,250300.0,1.650036e+07,42.948226,43.948226
208,undertone,500000.0,1.559987e+07,19.799830,20.799830
212,Backrooms,10000000.0,2.535098e+08,15.900655,16.900655
50,A Ghost Story,100000.0,1.397251e+06,8.315007,9.315007
154,Talk to Me,4500000.0,6.242043e+07,8.247471,9.247471
187,Ne Zha 2,80000000.0,1.359390e+09,7.496185,8.496185
142,The Whale,3000000.0,3.718893e+07,7.264206,8.264206
42,Moonlight,4000000.0,4.320625e+07,6.201042,7.201042
123,X,1000000.0,1.061245e+07,6.074967,7.074967
132,Pearl,1000000.0,7.322011e+06,3.881340,4.881341


In [8]:
df.to_csv("../processed/films_financial.csv", index=False)